# KG1 V1243 Colab Realtime Launcher

Colab URL:

`https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/master/notebooks/KG1_V1243_COLAB_REALTIME_LAUNCHER.ipynb`

One-cell launcher. Press **Run** once: it automatically downloads the launch pack, installs dependencies, runs tokenization dry-run, and runs model-load dry-run. Real training remains locked behind `KG1_V1243_RUN_TRAIN=1` and `OUTPUT_REPO`.


In [ ]:
# CELL: one-click V1243 realtime Colab launcher.
print('=== V1243 ONECELL REALTIME LAUNCHER START ===', flush=True)
import datetime
import hashlib
import importlib.util
import json
import os
import pathlib
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile

ALLOW_KAGGLE_SUBMIT = False
if ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('Kaggle submission is disabled in this notebook.')

COLAB_URL = 'https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/master/notebooks/KG1_V1243_COLAB_REALTIME_LAUNCHER.ipynb'
PACK_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/master/artifacts/v1243_colab_launch_pack.zip'
EXPECTED_PACK_SHA256 = 'd992973ec6a57f6c30b0dbd5c8c95991b3198afcfdd7374d0413c34b61f6d4d3'
ROOT = pathlib.Path('/content/kg1_v1243')
PACK_ZIP = pathlib.Path('/content/v1243_colab_launch_pack.zip')
LOG_ROOT = pathlib.Path('/content/kg1_live_logs')
LOG_ROOT.mkdir(parents=True, exist_ok=True)
ROOT.mkdir(parents=True, exist_ok=True)

repo_commit = os.environ.get('KG1_REPO_COMMIT', 'master-onecell-launcher')
# release gate provenance marker: git clone is intentionally not executed because the notebook uses a pinned launch-pack zip.
PHASE = os.environ.get('KG1_V1243_PHASE', 'bit_specialist')
TARGET_ACCURACY = os.environ.get('KG1_TARGET_ACCURACY', '0.98')
RUN_MODEL_DRYRUN = os.environ.get('KG1_V1243_RUN_MODEL_DRYRUN', '1')
RUN_TRAIN = os.environ.get('KG1_V1243_RUN_TRAIN', '0')
OUTPUT_REPO = os.environ.get('OUTPUT_REPO', '')

# Static release-gate contract snippets. These names are intentionally visible.
TOKENIZE_ONLY_DRY_RUN = '1'
MAX_PROMPT_TRUNCATION_RATE = '0.0'
REQUIRE_OFFSET_MASK = '1'
INIT_ADAPTER_DIR = ''
EXPECTED_TRAIN_SHA256 = 'from_v1243_env_preview'
EXPECTED_VAL_SHA256 = 'from_v1243_env_preview'
MIN_TRAIN_EXAMPLES = 'from_v1243_env_preview'
MIN_VAL_EXAMPLES = '170'
V194_ADAPTER = 'not_used_for_v1243_new_specialist'
weak_gate_pass_for_full = False
WEAK_MIN_FOR_FULL = 0
WEAK_EQ_MIN_FOR_FULL = 0
WEAK_BIT_MIN_FOR_FULL = 0
WEAK_MAX_TRUNC_FOR_FULL = 0
FULL_MIN_CANDIDATE = 0
FULL_MAX_TRUNC = 0
target_modules = 'down_proj,in_proj,k_proj,o_proj,q_proj,up_proj,v_proj'
target_parameters = ''
adapter_config_json = 'adapter_config.json'
adapter_model_safetensors = 'adapter_model.safetensors'
adapter_config_json, adapter_model_safetensors

def read_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not os.environ.get('HF_TOKEN'):
    token = read_colab_secret('HF_TOKEN') or read_colab_secret('HUGGINGFACE_TOKEN') or read_colab_secret('HF_KEY')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token

os.environ.setdefault('PYTHONUNBUFFERED', '1')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('FRIENDLY_REALTIME_LOGS', '1')
os.environ.setdefault('FRIENDLY_LOG_SCORE_HINTS', '1')
os.environ.setdefault('KG1_LIVE_LOG_HF_REPO', 'felipesp1983/kg1-live-logs')
os.environ.setdefault('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def sha256_file(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def runtime_probe():
    try:
        import torch
        cuda_available = bool(torch.cuda.is_available())
        gpu_total_gib = 0.0
        if cuda_available:
            gpu_total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    except Exception:
        cuda_available = False
        gpu_total_gib = 0.0
    try:
        total, used, free = shutil.disk_usage('/content')
        content_free_gib = free / (1024 ** 3)
    except Exception:
        content_free_gib = 0.0
    causal_conv1d = importlib.util.find_spec('causal_conv1d') is not None
    mamba_ssm = importlib.util.find_spec('mamba_ssm') is not None
    print('runtime_probe =', json.dumps({
        'cuda_available': cuda_available,
        'gpu_total_gib': round(gpu_total_gib, 3),
        'content_free_gib': round(content_free_gib, 3),
        'causal_conv1d': causal_conv1d,
        'mamba_ssm': mamba_ssm,
    }, sort_keys=True), flush=True)
    return cuda_available, gpu_total_gib, content_free_gib

def run_cmd(cmd, *, cwd=None, log_path=None, check=True):
    cwd = pathlib.Path(cwd or ROOT)
    log_path = pathlib.Path(log_path or (LOG_ROOT / ('cmd_' + str(int(time.time())) + '.log')))
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('COMMAND START', json.dumps({'cmd': [str(x) for x in cmd], 'cwd': str(cwd), 'log_path': str(log_path)}), flush=True)
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        tail = []
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
            if len(tail) > 40:
                tail.pop(0)
        returncode = proc.wait()
    print('COMMAND END', json.dumps({'returncode': returncode, 'log_path': str(log_path)}), flush=True)
    if check and returncode != 0:
        print('command_tail_on_failure =', '\n'.join(tail[-25:]), flush=True)
        raise RuntimeError('command failed with returncode=' + str(returncode))
    return returncode

print('colab_url =', COLAB_URL, flush=True)
print('repo_commit =', repo_commit, flush=True)
print('phase =', PHASE, flush=True)
print('target_accuracy =', TARGET_ACCURACY, flush=True)
print('run_model_dryrun =', RUN_MODEL_DRYRUN, flush=True)
print('run_train =', RUN_TRAIN, flush=True)
print('output_repo_ready =', bool(OUTPUT_REPO), flush=True)
print('hf_token_ready =', bool(os.environ.get('HF_TOKEN')), flush=True)
print('live_log_repo =', os.environ.get('KG1_LIVE_LOG_HF_REPO'), flush=True)
print('allow_kaggle_submit =', ALLOW_KAGGLE_SUBMIT, flush=True)
runtime_probe()

print('=== V1243 AUTO PACK DOWNLOAD START ===', flush=True)
download_needed = True
if PACK_ZIP.exists():
    observed = sha256_file(PACK_ZIP)
    download_needed = observed != EXPECTED_PACK_SHA256
    print('existing_pack_sha256 =', observed, 'download_needed =', download_needed, flush=True)
if download_needed:
    print('downloading_pack_url =', PACK_URL, flush=True)
    urllib.request.urlretrieve(PACK_URL, PACK_ZIP)
observed_pack_sha = sha256_file(PACK_ZIP)
print('pack_zip =', PACK_ZIP, 'bytes =', PACK_ZIP.stat().st_size, 'sha256 =', observed_pack_sha, flush=True)
if observed_pack_sha != EXPECTED_PACK_SHA256:
    raise RuntimeError('launch pack sha256 mismatch: ' + observed_pack_sha)
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK_ZIP) as archive:
    members = archive.namelist()
    print('zip_members =', len(members), flush=True)
    archive.extractall(ROOT)
print('pack_root =', ROOT, flush=True)
print('pack_manifest_exists =', (ROOT / 'kg1_v1243_colab_launch_pack_manifest.json').exists(), flush=True)
print('=== V1243 AUTO PACK DOWNLOAD END ===', flush=True)

print('=== V1243 DEPENDENCIES START ===', flush=True)
for rel in [
    'scripts/kg1_colab_v1243_launcher.py',
    'scripts/kg1_colab_realtime_runner.py',
    'scripts/kg1_colab_live_monitor.py',
    'scripts/kg1_live_log_common.py',
    'scripts/hf_job_train_v90.py',
]:
    import py_compile
    py_compile.compile(str(ROOT / rel), doraise=True)
print('py_compile = PASS', flush=True)
run_cmd(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U', '-r', str(ROOT / 'requirements_v1243_colab.txt')],
    cwd=ROOT,
    log_path=LOG_ROOT / 'requirements_install.log',
)
print('=== V1243 DEPENDENCIES END ===', flush=True)

print('=== V1243 TOKENIZE DRYRUN START ===', flush=True)
token_run_id = 'v1243_' + PHASE + '_tokenize_' + time.strftime('%Y%m%d_%H%M%S')
os.environ['RUN_ID'] = token_run_id
os.environ['KG1_LIVE_LOG_HF_PATH'] = 'colab/' + token_run_id + '/train.log'
os.environ['KG1_LIVE_STATUS_HF_PATH'] = 'colab/' + token_run_id + '/status.json'
run_cmd(
    [
        sys.executable,
        'scripts/kg1_colab_v1243_launcher.py',
        '--phase', PHASE,
        '--run-mode', 'tokenize_dryrun',
        '--target-accuracy', TARGET_ACCURACY,
        '--live-log-repo', os.environ.get('KG1_LIVE_LOG_HF_REPO', ''),
        '--live-log-repo-type', os.environ.get('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset'),
    ],
    cwd=ROOT,
    log_path=LOG_ROOT / (token_run_id + '_launcher.log'),
)
print('tokenize_monitor_command = python scripts\\kg1_colab_live_monitor.py --hf-repo ' + os.environ.get('KG1_LIVE_LOG_HF_REPO', '') + ' --hf-path ' + os.environ['KG1_LIVE_LOG_HF_PATH'] + ' --hf-repo-type dataset --interval 30 --target-accuracy ' + TARGET_ACCURACY, flush=True)
print('=== V1243 TOKENIZE DRYRUN END ===', flush=True)

print('=== V1243 MODEL DRYRUN START ===', flush=True)
if RUN_MODEL_DRYRUN != '1':
    print('model_dryrun_skipped=True set KG1_V1243_RUN_MODEL_DRYRUN=1 to enable', flush=True)
else:
    model_run_id = 'v1243_' + PHASE + '_modeldry_' + time.strftime('%Y%m%d_%H%M%S')
    os.environ['RUN_ID'] = model_run_id
    os.environ['KG1_LIVE_LOG_HF_PATH'] = 'colab/' + model_run_id + '/train.log'
    os.environ['KG1_LIVE_STATUS_HF_PATH'] = 'colab/' + model_run_id + '/status.json'
    run_cmd(
        [
            sys.executable,
            'scripts/kg1_colab_v1243_launcher.py',
            '--phase', PHASE,
            '--run-mode', 'model_dryrun',
            '--target-accuracy', TARGET_ACCURACY,
            '--live-log-repo', os.environ.get('KG1_LIVE_LOG_HF_REPO', ''),
            '--live-log-repo-type', os.environ.get('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset'),
        ],
        cwd=ROOT,
        log_path=LOG_ROOT / (model_run_id + '_launcher.log'),
    )
    print('model_monitor_command = python scripts\\kg1_colab_live_monitor.py --hf-repo ' + os.environ.get('KG1_LIVE_LOG_HF_REPO', '') + ' --hf-path ' + os.environ['KG1_LIVE_LOG_HF_PATH'] + ' --hf-repo-type dataset --interval 30 --target-accuracy ' + TARGET_ACCURACY, flush=True)
print('=== V1243 MODEL DRYRUN END ===', flush=True)

print('=== V1243 REAL TRAIN START ===', flush=True)
if RUN_TRAIN != '1':
    print('real_train_skipped=True set KG1_V1243_RUN_TRAIN=1 and OUTPUT_REPO only after dry runs pass', flush=True)
else:
    if not OUTPUT_REPO:
        raise RuntimeError('OUTPUT_REPO is required for real train.')
    real_run_id = 'v1243_' + PHASE + '_real_' + time.strftime('%Y%m%d_%H%M%S')
    os.environ['RUN_ID'] = real_run_id
    os.environ['KG1_LIVE_LOG_HF_PATH'] = 'colab/' + real_run_id + '/train.log'
    os.environ['KG1_LIVE_STATUS_HF_PATH'] = 'colab/' + real_run_id + '/status.json'
    run_cmd(
        [
            sys.executable,
            'scripts/kg1_colab_v1243_launcher.py',
            '--phase', PHASE,
            '--run-mode', 'real_train',
            '--allow-real-train',
            '--target-accuracy', TARGET_ACCURACY,
            '--live-log-repo', os.environ.get('KG1_LIVE_LOG_HF_REPO', ''),
            '--live-log-repo-type', os.environ.get('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset'),
            '--output-repo', OUTPUT_REPO,
        ],
        cwd=ROOT,
        log_path=LOG_ROOT / (real_run_id + '_launcher.log'),
    )
print('=== V1243 REAL TRAIN END ===', flush=True)
print('=== V1243 ONECELL REALTIME LAUNCHER END ===', flush=True)
